# **Statistical Learning for Automation Systems**

       
## *Prof. Simone Formentin*

___

# Classification with Neural Networks - Case Study

Let's consider an example of classification problem solved by using Neural Networks.

## Dataset Description

### Breast cancer Wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

    :Number of Instances: 569

    :Number of Attributes: 30 numeric, predictive attributes and the class

    :Attribute Information:
        - radius (mean of distances from center to points on the perimeter)
        - texture (standard deviation of gray-scale values)
        - perimeter
        - area
        - smoothness (local variation in radius lengths)
        - compactness (perimeter^2 / area - 1.0)
        - concavity (severity of concave portions of the contour)
        - concave points (number of concave portions of the contour)
        - symmetry 
        - fractal dimension ("coastline approximation" - 1)

        The mean, standard error, and "worst" or largest (mean of the three
        largest values) of these features were computed for each image,
        resulting in 30 features.  For instance, field 3 is Mean Radius, field
        13 is Radius SE, field 23 is Worst Radius.

        - class:
                - WDBC-Malignant
                - WDBC-Benign

    :Summary Statistics:

    ===================================== ====== ======
                                           Min    Max
    ===================================== ====== ======
    radius (mean):                        6.981  28.11
    texture (mean):                       9.71   39.28
    perimeter (mean):                     43.79  188.5
    area (mean):                          143.5  2501.0
    smoothness (mean):                    0.053  0.163
    compactness (mean):                   0.019  0.345
    concavity (mean):                     0.0    0.427
    concave points (mean):                0.0    0.201
    symmetry (mean):                      0.106  0.304
    fractal dimension (mean):             0.05   0.097
    radius (standard error):              0.112  2.873
    texture (standard error):             0.36   4.885
    perimeter (standard error):           0.757  21.98
    area (standard error):                6.802  542.2
    smoothness (standard error):          0.002  0.031
    compactness (standard error):         0.002  0.135
    concavity (standard error):           0.0    0.396
    concave points (standard error):      0.0    0.053
    symmetry (standard error):            0.008  0.079
    fractal dimension (standard error):   0.001  0.03
    radius (worst):                       7.93   36.04
    texture (worst):                      12.02  49.54
    perimeter (worst):                    50.41  251.2
    area (worst):                         185.2  4254.0
    smoothness (worst):                   0.071  0.223
    compactness (worst):                  0.027  1.058
    concavity (worst):                    0.0    1.252
    concave points (worst):               0.0    0.291
    symmetry (worst):                     0.156  0.664
    fractal dimension (worst):            0.055  0.208
    ===================================== ====== ======

    :Missing Attribute Values: None

    :Class Distribution: 212 - Malignant, 357 - Benign

    :Creator:  Dr. William H. Wolberg, W. Nick Street, Olvi L. Mangasarian

    :Donor: Nick Street

    :Date: November, 1995

This is a copy of UCI ML Breast Cancer Wisconsin (Diagnostic) datasets.
https://goo.gl/U2Uwz2

Features are computed from a digitized image of a fine needle
aspirate (FNA) of a breast mass.  They describe
characteristics of the cell nuclei present in the image.

Separating plane described above was obtained using
Multisurface Method-Tree (MSM-T) [K. P. Bennett, "Decision Tree
Construction Via Linear Programming." Proceedings of the 4th
Midwest Artificial Intelligence and Cognitive Science Society,
pp. 97-101, 1992], a classification method which uses linear
programming to construct a decision tree.  Relevant features
were selected using an exhaustive search in the space of 1-4
features and 1-3 separating planes.

The actual linear program used to obtain the separating plane
in the 3-dimensional space is that described in:
[K. P. Bennett and O. L. Mangasarian: "Robust Linear
Programming Discrimination of Two Linearly Inseparable Sets",
Optimization Methods and Software 1, 1992, 23-34].

This database is also available through the UW CS ftp server:

ftp ftp.cs.wisc.edu
cd math-prog/cpo-dataset/machine-learn/WDBC/

.. topic:: References

   - W.N. Street, W.H. Wolberg and O.L. Mangasarian. Nuclear feature extraction 
     for breast tumor diagnosis. IS&T/SPIE 1993 International Symposium on 
     Electronic Imaging: Science and Technology, volume 1905, pages 861-870,
     San Jose, CA, 1993.
   - O.L. Mangasarian, W.N. Street and W.H. Wolberg. Breast cancer diagnosis and 
     prognosis via linear programming. Operations Research, 43(4), pages 570-577, 
     July-August 1995.
   - W.H. Wolberg, W.N. Street, and O.L. Mangasarian. Machine learning techniques
     to diagnose breast cancer from fine-needle aspirates. Cancer Letters 77 (1994) 
     163-171.

## Import Data Analysis and Visualization Libraries

In [ ]:
%pip install numpy pandas matplotlib seaborn openpyxl
%pip install ipywidgets

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

## Import Dataset

In [ ]:
df = pd.read_csv('cancer_classification.csv')          # Import Dataset from .csv file
df.rename(columns={"benign_0__mal_1" : "benign_1__mal_0"}, inplace = True)          # Rename column
df.head()                                              # Show header

In [ ]:
df.info()

In [ ]:
df.describe().transpose()

## Exploratory Data Analysis

In [ ]:
# Plot number of benign vs malignant
sns.countplot(x='benign_1__mal_0',data=df,hue='benign_1__mal_0')

# Plot correlation heatmap
plt.figure(figsize=(12,8))                                 
sns.heatmap(df.corr())                    

# Reorder correlation values for easier inspection
df.corr()['benign_1__mal_0'].sort_values(ascending = False)

# Plot these values
plt.figure(figsize=(10,3))                                        # Create a figure object with a specified figure size
df.corr()['benign_1__mal_0'].sort_values().plot(kind='bar')       # Barplot for the sorted correlation coefficients

## Train/Test Set Split

In [ ]:
from sklearn.model_selection import train_test_split            # Import train_test_split function from Scikit-Learn

# Features
X = df.drop('benign_1__mal_0',axis=1).values

# Target Variable
y = df['benign_1__mal_0'].values

# Perform split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=101)         # Train-Test Split
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

## Data Pre-Processing

Use StandardScaler() to perform input centering and input normalization.

Notice that it is sufficient to standardize just input features, while it is not necessary to standardize the target variable.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Train scaler
scaler = StandardScaler()
scaler.fit(X_train)

# Apply scaling
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

## Neural Network Definition - TensorFlow & Keras

## Creating NN Model

In [ ]:
import tensorflow as tf

from tensorflow.keras.models import Sequential                         # Import Sequential() to instatiate a Feed-forward NN object
from tensorflow.keras.layers import Dense                              # Import Dense() to instatiate fully-connected layer objects

In [ ]:
# Generate model
model = Sequential()
model.add(Dense(units=30,activation='relu'))
model.add(Dense(units=15,activation='relu'))

# Final output node for prediction
model.add(Dense(units=1, activation='sigmoid'))                         # In the output layer we just have one neuron since the target is a single variable (class)
 
# For a binary classification problem
model.compile(loss='binary_crossentropy', optimizer='adam')             # Optimizer determines the way in which you approach gradient descent (alternative: ADAM)

### Choosing suitable Loss functions and Evaluation Metrics

Keep in mind what kind of problem you are trying to solve:

    # For a multi-class classification problem
    model.compile(optimizer='rmsprop',
                  loss='categorical_crossentropy',
                  metrics='accuracy')

    # For a binary classification problem
    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics='accuracy')

    # For a mean squared error regression problem
    model.compile(optimizer='rmsprop',
                  loss='mse')

## Training the Model 

### Example One

In [ ]:
model.fit(x=X_train, y=y_train, epochs=1000, validation_data=(X_test, y_test), verbose=1)

In [ ]:
# Get information about the model
model.summary()

# Save Training and Validation Loss in a Pandas DataFrame
model_loss = pd.DataFrame(model.history.history)    
model_loss.head()

In [ ]:
# Plot Training / Validation Loss vs Epochs
plt.figure(figsize = (8, 4))

plt.plot(range(len(model_loss)), model_loss['loss'], label = 'Training Loss')
plt.plot(range(len(model_loss)), model_loss['val_loss'], label = 'Validation Loss')

plt.xlim([-10, len(model_loss)])
plt.xlabel('Epoch')
plt.ylabel('Training / Validation Loss')
plt.title('Training / Validation Loss vs Epochs')
plt.legend()
plt.grid()

Choosing too many epochs and overfitting!

## Example Two: Early Stopping

We obviously trained too much! Let's use early stopping to track the val_loss and stop training once it begins increasing too much!

In [ ]:
# Train again a new model
model = Sequential()
model.add(Dense(units=30,activation='relu'))
model.add(Dense(units=15,activation='relu'))
model.add(Dense(units=1,activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam')

Stop training when a monitored quantity has stopped improving.

    Arguments:
        monitor: Quantity to be monitored.
        min_delta: Minimum change in the monitored quantity
            to qualify as an improvement, i.e. an absolute
            change of less than min_delta, will count as no
            improvement.
        patience: Number of epochs with no improvement
            after which training will be stopped.
        verbose: verbosity mode.
        mode: One of `{"auto", "min", "max"}`. In `min` mode,
            training will stop when the quantity
            monitored has stopped decreasing; in `max`
            mode it will stop when the quantity
            monitored has stopped increasing; in `auto`
            mode, the direction is automatically inferred
            from the name of the monitored quantity.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping                 # Import EarlyStopping method

# Add early stopping
early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=25)       # Define EarlyStopping object and assign it to variable "early_stop"

# Fit with early stopping
model.fit(x=X_train, y=y_train, epochs=600, validation_data=(X_test, y_test), verbose=1, callbacks=[early_stop])

# Save results
model_loss = pd.DataFrame(model.history.history)           # Save Training and Validation Loss in a Pandas DataFrame 

In [ ]:
# Plot Training / Validation Loss vs Epochs

plt.figure(figsize = (10, 4))

plt.plot(range(len(model_loss)), model_loss['loss'], label = 'Training Loss')
plt.plot(range(len(model_loss)), model_loss['val_loss'], label = 'Validation Loss')

plt.xlim([-10, len(model_loss)])
plt.xlabel('Epoch')
plt.ylabel('Training / Validation Loss')
plt.title('Training / Validation Loss vs Epochs')
plt.legend()
plt.grid()

## Example Three: Adding in DropOut Layers

In [ ]:
from tensorflow.keras.layers import Dropout                     # Import Dropout method

# Get information about the class
help(Dropout)

In [ ]:
# Train again the new model
model = Sequential()

model.add(Dense(units=30,activation='relu'))
model.add(Dropout(0.5))                                    # Add Dropout option to NN layers

model.add(Dense(units=15,activation='relu'))
model.add(Dropout(0.5))                                    # Add Dropout option to NN layers
  
model.add(Dense(units=1,activation='sigmoid'))
          
model.compile(loss='binary_crossentropy', optimizer='adam')

In [ ]:
model.fit(x=X_train, y=y_train, epochs=600, validation_data=(X_test, y_test), verbose=1, callbacks=[early_stop])

In [ ]:
# Get information about model
model.summary()

# Save Training and Validation Loss in a Pandas DataFramemodel_loss.plot()
model_loss = pd.DataFrame(model.history.history)    

# Plot Training / Validation Loss vs Epochs
plt.figure(figsize = (8, 4))
plt.plot(range(len(model_loss)), model_loss['loss'], label = 'Training Loss')
plt.plot(range(len(model_loss)), model_loss['val_loss'], label = 'Validation Loss')
plt.xlim([0, len(model_loss)])
plt.xlabel('Epoch')
plt.ylabel('Training / Validation Loss')
plt.title('Training / Validation Loss vs Epochs')
plt.legend()
plt.grid()

# Model Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix             # Import model evaluation Metrics

# Get predictions and set them to either 1 or 0
predictions = model.predict(X_test)
predictions = np.where(predictions > 0.5, 1, 0)

# Print results
print(classification_report(y_test, predictions))
print('Confusion matrix')
print(confusion_matrix(y_test, predictions))